In [49]:
import pandas as pd
import numpy as np
import geo.sphere
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
# Selected GTFS shapes within and intersecting Liverpool LGA polygon
lvplshapes = pd.read_csv('./input/LiverpoolRoutes.csv')
lvplshapes.head()

,shape_id,begin,end
0,21-857-sj2-1.9.R,1,519
1,21-852-sj2-1.9.H,1,1031
2,21-852-sj2-1.8.R,1,545
3,21-852-sj2-1.7.H,1,584
4,21-852-sj2-1.6.R,1,716


In [4]:
len(lvplshapes)

949

In [5]:
trips = pd.read_csv("./input/gtfs_static_8jul2020/trips.txt")
stoptimes = pd.read_csv("./input/gtfs_static_8jul2020/stop_times.txt")
routes = pd.read_csv("./input/gtfs_static_8jul2020/routes.txt")
stops = pd.read_csv("./input/gtfs_static_8jul2020/stops.txt")
shapes = pd.read_csv("./input/gtfs_static_8jul2020/shapes.txt")

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0,3,5,10) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


### Get trips

In [6]:
trips.head()

,route_id,service_id,trip_id,shape_id,trip_headsign,direction_id,block_id,wheelchair_accessible,route_direction,trip_note,bikes_allowed
0,1-SC0-1-sj2-3,AA51+1,1.AA51.1-SC0-1-sj2-3.1.R,1-SC0-1-sj2-3.1.R,Kiama,1,NaN,1,Bomaderry to Kiama,NaN,NaN
1,1-SC0-1-sj2-3,AA51+1,3.AA51.1-SC0-1-sj2-3.1.R,1-SC0-1-sj2-3.1.R,Kiama,1,NaN,1,Bomaderry to Kiama,NaN,NaN
2,1-SC0-1-sj2-3,AA51+1,5.AA51.1-SC0-1-sj2-3.1.R,1-SC0-1-sj2-3.1.R,Kiama,1,NaN,1,Bomaderry to Kiama,NaN,NaN
3,1-SC0-1-sj2-3,AA51+1,7.AA51.1-SC0-1-sj2-3.2.H,1-SC0-1-sj2-3.2.H,Bomaderry,0,NaN,1,Kiama to Bomaderry,NaN,NaN
4,1-SC0-1-sj2-3,AA51+1,9.AA51.1-SC0-1-sj2-3.2.H,1-SC0-1-sj2-3.2.H,Bomaderry,0,NaN,1,Kiama to Bomaderry,NaN,NaN


In [8]:
trips['shape_id'] = trips['shape_id'].astype(str)
lvplshapes['shape_id'] = lvplshapes['shape_id'].astype(str)
print(len(trips))
lvpltrips = trips.loc[trips['shape_id'].isin(lvplshapes['shape_id'])]
print(len(lvpltrips))

166978
12623


### Check and keep only bus routes

In [24]:
lvpltrips['route_id'] = lvpltrips['route_id'].astype(str)
routes['route_id'] = routes['route_id'].astype(str)
lvpltrips = lvpltrips.merge(routes, how='left', on='route_id')
len(lvpltrips)

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


12623

In [26]:
lvpltrips['route_desc'].unique()

array(['Temporary buses', 'Sydney Trains Network', 'Sydney Buses Network',
       'School buses', 'Regional Trains and Coaches Network',
       'Intercity Trains Network'], dtype=object)

In [27]:
lvpltrips = lvpltrips.loc[lvpltrips['route_desc']=='Sydney Buses Network']
len(lvpltrips)

3673

In [29]:
len(lvpltrips['route_id'].unique())

42

### Get travel time, stops and route/shape length

In [21]:
stoptimes.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,stop_note
0,2.AA70.1-10H-U-sj2-2.1.R,10:09:00,10:09:00,233312,1,NaN,0,0,0.00,1,NaN
1,2.AA70.1-10H-U-sj2-2.1.R,10:52:00,10:52:00,233011,2,NaN,0,0,50450.60,1,NaN
2,2.AA70.1-10H-U-sj2-2.1.R,11:34:00,11:34:00,232021,3,NaN,0,0,97416.41,1,NaN
3,1.AA70.1-10H-U-sj2-2.2.H,08:50:00,08:50:00,232021,1,NaN,0,0,0.00,1,NaN
4,1.AA70.1-10H-U-sj2-2.2.H,09:35:00,09:35:00,233012,2,NaN,0,0,47346.57,1,NaN


In [30]:
lvpltrips['trip_id'] = lvpltrips['trip_id'].astype(str)
stoptimes['trip_id'] = stoptimes['trip_id'].astype(str)
print(len(stoptimes))
lvplstoptimes = stoptimes.loc[stoptimes['trip_id'].isin(lvpltrips['trip_id'])]
print(len(lvplstoptimes))

4624043
215337


In [31]:
len(lvplstoptimes['trip_id'].unique())

3673

In [38]:
lvpldf = lvplstoptimes[['trip_id','arrival_time']].loc[lvplstoptimes['stop_sequence']==1].reset_index(drop=True)
lvpldf = lvpldf.merge(lvplstoptimes[['trip_id','departure_time','stop_sequence','shape_dist_traveled']].groupby('trip_id').max().reset_index(), how='left', on='trip_id')
lvpldf.columns = ['trip_id','start_time','end_time','stops','shape_dist_traveled']
lvpldf['start_hr'] = lvpldf['start_time'].apply(lambda x: int(x.split(':')[0]))
lvpldf['end_hr'] = lvpldf['end_time'].apply(lambda x: int(x.split(':')[0]))
print(len(lvpldf))
lvpldf = lvpldf.loc[(lvpldf['start_hr']<24) & (lvpldf['end_hr']<24)]
print(len(lvpldf))
lvpldf['start_time'] = pd.to_datetime(lvpldf['start_time'],format = '%H:%M:%S')
lvpldf['end_time'] = pd.to_datetime(lvpldf['end_time'],format = '%H:%M:%S')
lvpldf['travel_time_h'] = (lvpldf['end_time'] - lvpldf['start_time'])/np.timedelta64(1,'h')


3673
3627


In [41]:
lvpldf = lvpldf[['trip_id','travel_time_h','stops','shape_dist_traveled']]
lvpldf.columns = ['trip_id','travel_time_h','stops','dist_traveled']
lvpldf.head()

,trip_id,travel_time_h,stops,dist_traveled
0,987723,0.500000,20,18114.67
1,1028426,0.683333,20,20623.03
2,1162320,0.500000,22,18928.33
3,1135016,0.566667,18,18866.30
4,1135008,0.500000,17,18969.40


### Travel time model

In [77]:
lvpldf['dist_traveled'] = lvpldf['dist_traveled']/1000
lvpldf.head()

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,trip_id,travel_time_h,stops,dist_traveled
0,987723,0.500000,20,18.11467
1,1028426,0.683333,20,20.62303
2,1162320,0.500000,22,18.92833
3,1135016,0.566667,18,18.86630
4,1135008,0.500000,17,18.96940


In [79]:
#lvplmodel = smf.ols('travel_time_h ~ dist_traveled + stops', data = lvpldf).fit()
lvplmodel = smf.ols('travel_time_h ~ dist_traveled + stops - 1', data = lvpldf).fit() # Without intercept
lvplmodel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:          travel_time_h   R-squared (uncentered):                   0.988
Model:                            OLS   Adj. R-squared (uncentered):              0.988
Method:                 Least Squares   F-statistic:                          1.534e+05
Date:                Thu, 13 Aug 2020   Prob (F-statistic):                        0.00
Time:                        10:55:51   Log-Likelihood:                          2925.0
No. Observations:                3627   AIC:                                     -5846.
Df Residuals:                    3625   BIC:                                     -5834.
Df Model:                           2                                                  
Covariance Type:            nonrobust                                                  
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
dist_traveled     0.0262      0.000    116.143      0.000       0.026       0.027
stops             0.0055   8.03e-05     68.979      0.000       0.005       0.006
==============================================================================
Omnibus:                       89.685   Durbin-Watson:                   0.517
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              196.673
Skew:                          -0.095   Prob(JB):                     1.96e-43
Kurtosis:                       4.125   Cond. No.                         9.48
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [80]:
1/0.0262

38.167938931297705

In [81]:
0.0055*3600

19.799999999999997

In [83]:
lvpldf['speed_kmph'] = lvpldf['dist_traveled']/lvpldf['travel_time_h']
lvpldf.describe()

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,travel_time_h,stops,dist_traveled,speed_kmph
count,3627.000000,3627.000000,3627.000000,3627.000000
mean,0.904094,59.035291,21.974932,24.688363
std,0.426683,33.164441,10.021573,4.450262
min,0.083333,4.000000,1.849610,13.122309
25%,0.533333,35.000000,12.737330,21.657900
50%,0.883333,46.000000,21.245000,23.804874
75%,1.250000,94.000000,30.720500,27.513754
max,2.150000,133.000000,59.455500,54.852720


### Separate tway trips

In [42]:
twaytrips = lvpltrips.loc[lvpltrips['route_id'].str.contains('T80')]
nontwaytrips = lvpltrips.loc[lvpltrips['route_id'].str.contains('T80')==False]
print(len(twaytrips)+len(nontwaytrips))

3673


In [44]:
lvpldf['trip_id'] = lvpldf['trip_id'].astype(str)
twaytrips['trip_id'] = twaytrips['trip_id'].astype(str)
twaydf = lvpldf.loc[lvpldf['trip_id'].isin(twaytrips['trip_id'])]
len(twaydf)

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


563

In [46]:
lvpldf['trip_id'] = lvpldf['trip_id'].astype(str)
nontwaytrips['trip_id'] = nontwaytrips['trip_id'].astype(str)
nontwaydf = lvpldf.loc[lvpldf['trip_id'].isin(nontwaytrips['trip_id'])]
len(nontwaydf)

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


3064

### Travel time model - Tway trips

In [53]:
twaydf['dist_traveled'] = twaydf['dist_traveled']/1000
twaydf.head()

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,trip_id,travel_time_h,stops,dist_traveled
1389,1049990,1.033333,35,30.5571
1390,1049991,1.033333,35,30.5571
1391,1049992,1.033333,35,30.5571
1392,1049993,1.033333,35,30.5571
1393,1049994,1.033333,35,30.5571


In [82]:
#twaymodel = smf.ols('travel_time_h ~ dist_traveled + stops', data = twaydf).fit()
twaymodel = smf.ols('travel_time_h ~ dist_traveled + stops - 1', data = twaydf).fit() # Without intercept
twaymodel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:          travel_time_h   R-squared (uncentered):                   0.996
Model:                            OLS   Adj. R-squared (uncentered):              0.996
Method:                 Least Squares   F-statistic:                          7.798e+04
Date:                Thu, 13 Aug 2020   Prob (F-statistic):                        0.00
Time:                        10:59:52   Log-Likelihood:                          799.32
No. Observations:                 563   AIC:                                     -1595.
Df Residuals:                     561   BIC:                                     -1586.
Df Model:                           2                                                  
Covariance Type:            nonrobust                                                  
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
dist_traveled    -0.0362      0.012     -3.030      0.003      -0.060      -0.013
stops             0.0613      0.010      5.863      0.000       0.041       0.082
==============================================================================
Omnibus:                        4.205   Durbin-Watson:                   0.403
Prob(Omnibus):                  0.122   Jarque-Bera (JB):                4.880
Skew:                          -0.072   Prob(JB):                       0.0871
Kurtosis:                       3.433   Cond. No.                         281.
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [72]:
 1/0.0067

149.2537313432836

In [73]:
0.0222*3600

79.92

In [74]:
0.0511*3600

183.96

In [75]:
twaydf['speed_kmph'] = twaydf['dist_traveled']/twaydf['travel_time_h']
twaydf.describe()

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,travel_time_h,stops,dist_traveled,speed_kmph
count,563.000000,563.000000,563.000000,563.000000
mean,0.954056,32.158082,28.085614,29.328734
std,0.211635,7.206775,6.463739,2.141494
min,0.350000,13.000000,10.832280,24.071733
25%,0.966667,35.000000,30.557100,28.647281
50%,1.033333,35.000000,30.557100,29.571387
75%,1.066667,35.000000,30.720500,30.557100
max,1.183333,35.000000,30.720500,35.446731


### Travel time model - Non-tway trips

In [55]:
nontwaydf['dist_traveled'] = nontwaydf['dist_traveled']/1000
nontwaydf.head()

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,trip_id,travel_time_h,stops,dist_traveled
0,987723,0.500000,20,18.11467
1,1028426,0.683333,20,20.62303
2,1162320,0.500000,22,18.92833
3,1135016,0.566667,18,18.86630
4,1135008,0.500000,17,18.96940


In [66]:
#nontwaymodel = smf.ols('travel_time_h ~ dist_traveled + stops', data = nontwaydf).fit()
nontwaymodel = smf.ols('travel_time_h ~ dist_traveled + stops - 1', data = nontwaydf).fit() # Without intercept
nontwaymodel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:          travel_time_h   R-squared (uncentered):                   0.988
Model:                            OLS   Adj. R-squared (uncentered):              0.988
Method:                 Least Squares   F-statistic:                          1.299e+05
Date:                Thu, 13 Aug 2020   Prob (F-statistic):                        0.00
Time:                        10:39:23   Log-Likelihood:                          2462.1
No. Observations:                3064   AIC:                                     -4920.
Df Residuals:                    3062   BIC:                                     -4908.
Df Model:                           2                                                  
Covariance Type:            nonrobust                                                  
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
dist_traveled     0.0175      0.001     34.920      0.000       0.017       0.019
stops             0.0082      0.000     51.027      0.000       0.008       0.009
==============================================================================
Omnibus:                      101.453   Durbin-Watson:                   0.609
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              127.920
Skew:                           0.377   Prob(JB):                     1.67e-28
Kurtosis:                       3.658   Cond. No.                         20.4
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [69]:
1/0.0175

57.14285714285714

In [70]:
0.0082*3600

29.520000000000003

In [76]:
nontwaydf['speed_kmph'] = nontwaydf['dist_traveled']/nontwaydf['travel_time_h']
nontwaydf.describe()

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


,travel_time_h,stops,dist_traveled,speed_kmph
count,3064.000000,3064.000000,3064.000000,3064.000000
mean,0.894914,63.973890,20.852115,23.835710
std,0.454710,33.694795,10.153898,4.233023
min,0.083333,4.000000,1.849610,13.122309
25%,0.516667,36.000000,12.452000,21.320865
50%,0.750000,52.000000,17.027030,23.181189
75%,1.350000,101.000000,29.812820,25.408091
max,2.150000,133.000000,59.455500,54.852720
